In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType, DoubleType

In [2]:
# Initialize Spark Session
spark = SparkSession.builder.appName("LungCancerAnalysis").getOrCreate()

# Load the dataset
# Note: /content/Lung Cancer.csv is already extracted in the environment
df = spark.read.csv('/content/Lung Cancer.csv', header=True, inferSchema=True)
display(df.limit(5).toPandas())

,id,age,gender,country,diagnosis_date,cancer_stage,family_history,smoking_status,bmi,cholesterol_level,hypertension,asthma,cirrhosis,other_cancer,treatment_type,end_treatment_date,survived
0,1,64.0,Male,Sweden,2016-04-05,Stage I,Yes,Passive Smoker,29.4,199,0,0,1,0,Chemotherapy,2017-09-10,0
1,2,50.0,Female,Netherlands,2023-04-20,Stage III,Yes,Passive Smoker,41.2,280,1,1,0,0,Surgery,2024-06-17,1
2,3,65.0,Female,Hungary,2023-04-05,Stage III,Yes,Former Smoker,44.0,268,1,1,0,0,Combined,2024-04-09,0
3,4,51.0,Female,Belgium,2016-02-05,Stage I,No,Passive Smoker,43.0,241,1,1,0,0,Chemotherapy,2017-04-23,0
4,5,37.0,Male,Luxembourg,2023-11-29,Stage I,No,Passive Smoker,19.7,178,0,0,0,0,Combined,2025-01-08,0


### Task 1: Data Cleaning and Transformation
We'll create a function to remove duplicates, cast types, and convert 'yes'/'no' values.

In [5]:
def clean_data(df):
    # Remove duplicates
    df = df.dropDuplicates()

    # List of binary columns to convert
    binary_cols = [c for c, t in df.dtypes if t == 'string' and df.select(c).filter(F.col(c).rlike('(?i)^(yes|no)$')).count() > 0]

    for col in binary_cols:
        df = df.withColumn(col, F.when(F.lower(F.col(col)) == 'yes', 1).otherwise(0))

    # Ensure correct data types (Dates and Numerics)
    # Assuming standard column names based on the task description
    if 'Diagnosis_Date' in df.columns:
        df = df.withColumn('Diagnosis_Date', F.col('Diagnosis_Date').cast(DateType()))
    if 'end_treatment_date' in df.columns:
        df = df.withColumn('end_treatment_date', F.col('end_treatment_date').cast(DateType()))
    if 'Age' in df.columns:
        df = df.withColumn('Age', F.col('Age').cast(IntegerType()))
    if 'BMI' in df.columns:
        df = df.withColumn('BMI', F.col('BMI').cast(DoubleType()))

    return df

cleaned_df = clean_data(df)
cleaned_df.cache()
print("Data cleaned and duplicates removed.")

Data cleaned and duplicates removed.


### Task 2: Treatment Duration Analysis
Calculate days between diagnosis and treatment end, then average by treatment type.

In [6]:
def avg_treatment_duration(df):
    df_duration = df.withColumn('treatment_duration_days',
                                F.datediff(F.col('end_treatment_date'), F.col('Diagnosis_Date')))

    avg_duration = df_duration.groupBy('Treatment_Type').agg(F.avg('treatment_duration_days').alias('avg_duration'))
    return avg_duration

result_2 = avg_treatment_duration(cleaned_df)
display(result_2.toPandas())

,Treatment_Type,avg_duration
0,Radiation,458.403205
1,Chemotherapy,458.395401
2,Combined,457.815219
3,Surgery,457.737446


### Task 3: Highest Survival Rate by Smoking Status

In [7]:
def top_smoking_survival(df):
    # Assuming 'Survived' is 1 for yes, 0 for no
    survival_stats = df.groupBy('Smoking_Status').agg(F.avg('Survived').alias('survival_rate'))
    top_group = survival_stats.orderBy(F.desc('survival_rate')).first()
    return top_group

result_3 = top_smoking_survival(cleaned_df)
print(f"Highest survival rate group: {result_3['Smoking_Status']} ({result_3['survival_rate']:.2%})")

Highest survival rate group: Never Smoked (22.09%)


### Task 4: Top 3 Countries for Stage IV Diagnoses

In [8]:
def top_stage_iv_countries(df):
    country_counts = df.groupBy('Country').agg(
        F.count('*').alias('total_patients'),
        F.sum(F.when(F.col('Cancer_Stage') == 'Stage IV', 1).otherwise(0)).alias('stage_iv_count')
    )

    country_perc = country_counts.withColumn('percentage_stage_iv', (F.col('stage_iv_count') / F.col('total_patients')) * 100)
    return country_perc.orderBy(F.desc('percentage_stage_iv')).limit(3)

result_4 = top_stage_iv_countries(cleaned_df)
display(result_4.toPandas())

,Country,total_patients,stage_iv_count,percentage_stage_iv
0,Greece,33052,8429,25.502239
1,Croatia,33138,8426,25.427002
2,Czech Republic,32885,8317,25.291166


### Task 5: High-Risk Patient Filter
Filtering based on specific health and demographic criteria.

In [9]:
def filter_complex_patients(df):
    filtered = df.filter(
        (F.col('Gender') == 'Male') &
        (F.col('Cancer_Stage').isin(['Stage III', 'Stage IV'])) &
        (F.col('Family_History') == 1) &
        (F.col('Smoking_Status') == 'Current Smoker') &
        (F.col('BMI') > 30) &
        (F.col('Survived') == 1)
    )

    metrics = filtered.agg(
        F.avg('Age').alias('avg_age'),
        (F.avg('Hypertension') * 100).alias('hypertension_percentage')
    )
    return metrics

result_5 = filter_complex_patients(cleaned_df)
display(result_5.toPandas())

,avg_age,hypertension_percentage
0,55.179399,74.765185
